# 03 — KnottedGraph vs Topoly: paper-quality Yamada scaling

This notebook benchmarks the **Yamada engines only**. Spatial-graph construction and graph-to-PD conversion are performed **outside the timed region**; KnottedGraph and Topoly receive the same PD code for every individual sample.

Two complementary benchmark classes are run:

1. **Controlled families** isolate specific complexity axes such as projected crossings, edge count, connected size, and factorization. At each x value, 10 deterministic geometric embeddings are generated.
2. **Heterogeneous connected cubic graphs** use 10 pairwise **non-isomorphic** connected 3-regular graph topologies at every vertex count \(V\). This measures instance-to-instance variability rather than repeatedly drawing the same abstract graph.

For the paper profile, every plotted point is the median across the 10 independent sample-level timings and the shaded region is a deterministic nonparametric **95% bootstrap confidence interval**. Timing repetitions within one sample are reduced to a single median before the sample enters the statistical analysis.

Long computations stream their output live and display progress bars. Timeouts are treated as censored observations; they are not substituted into confidence intervals.


In [ ]:
from pathlib import Path
import csv, importlib.util, json, os, subprocess, sys
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "knotted_graph").exists():
    raise RuntimeError("Run this notebook from inside the KnottedGraph checkout.")

SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

import knotted_graph
kg_path = Path(knotted_graph.__file__).resolve()
if SRC not in kg_path.parents:
    raise RuntimeError(f"A stale knotted_graph was imported from {kg_path}")

try:
    import topoly
except ImportError as exc:
    raise ImportError("Install Topoly first: pip install topoly") from exc

OUT = ROOT / "User_guide" / "benchmarks"
RES = OUT / "results_latest"
FIG = OUT / "figures_latest"
RES.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

print("KnottedGraph:", kg_path)
print("Topoly:", Path(topoly.__file__).resolve())


## 1. Long-run configuration

A normal/local execution uses the **paper** configuration:

- 10 controlled embeddings per x value;
- 10 non-isomorphic connected cubic graph instances per \(V\);
- up to 120 s per framework per sample;
- broad x-axis ranges intended for publication-quality scaling figures.

This can run for many hours. GitHub Actions sets `CI=true`; only there the notebook automatically switches to a small smoke configuration. **Do not use smoke-mode output in a paper.**


In [ ]:
IS_CI = os.environ.get("CI", "").lower() == "true"

PROFILE = "smoke" if IS_CI else "paper"
SAMPLES_PER_X = 2 if IS_CI else 10
TIMEOUT_S = 10 if IS_CI else 120
BASE_SEED = 20260818

raw_csv = RES / "topoly_yamada_scaling_raw.csv"
aggregate_csv = RES / "topoly_yamada_scaling_aggregate.csv"

print(
    f"mode={PROFILE}, samples/x={SAMPLES_PER_X}, "
    f"timeout/framework/sample={TIMEOUT_S}s"
)


## 2. Live progress helper

Each completed JSON benchmark row corresponds to **one independent sample at one x value**. The progress bar therefore advances only after both framework attempts for that sample have finished (or been censored). The postfix reports the current family, \(V/E/c\) metadata when available, sample number, and KnottedGraph/Topoly status.


In [ ]:
def _load_module(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not load {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


def run_streamed_benchmark(cmd, *, total, description):
    print("Running:", " ".join(map(str, cmd)))
    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Benchmark subprocess did not expose stdout.")

    streamed_rows = []
    summary_rows = None
    bar = tqdm(total=total, desc=description, unit="sample", dynamic_ncols=True)

    for raw_line in process.stdout:
        line = raw_line.rstrip()
        if not line:
            continue

        if line.startswith("SUMMARY="):
            summary_rows = json.loads(line[len("SUMMARY="):])
            continue

        if line.startswith("{"):
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                tqdm.write(line)
                continue
            if "family" in row:
                streamed_rows.append(row)
                sample_id = row.get("sample", row.get("embedding", "?"))
                meta = []
                for key in ("V", "E", "crossings"):
                    if row.get(key) is not None:
                        meta.append(f"{key}={row[key]}")
                status = (
                    f"KG={row.get('knottedgraph_status','?')},"
                    f"T={row.get('topoly_status','?')}"
                )
                bar.set_postfix_str(
                    f"{row.get('family','?')} {' '.join(meta)} "
                    f"sample={sample_id} {status}"
                )
                bar.update(1)
                continue

        if line.startswith(("FAMILY=", "CENSOR_FRONTIER=", "CONFIG=")):
            tqdm.write(line)
        else:
            tqdm.write(line)

    return_code = process.wait()
    bar.close()
    if return_code:
        raise RuntimeError(
            f"Benchmark failed with exit code {return_code}: {' '.join(map(str, cmd))}"
        )

    rows = summary_rows if summary_rows is not None else streamed_rows
    if not rows:
        raise RuntimeError("Benchmark completed without sample rows.")
    print(f"completed {len(rows)} sample records")
    return rows


## 3. Controlled-family benchmark

These families are deliberately structured stress tests. They isolate crossing count, edge count, disconnected factorization, or connected trivalent size. The 10 samples at each x value vary their geometry while preserving the intended family constraints.


In [ ]:
controlled_script = ROOT / "dev" / "benchmark_topoly_extended_scaling.py"
env = dict(os.environ)
env["PYTHONPATH"] = str(SRC)
env["PYTHONNOUSERSITE"] = "1"

controlled_module = _load_module(
    controlled_script,
    "kg_topoly_scaling_notebook_plan",
)
controlled_plan = controlled_module._cases(PROFILE)
controlled_total = (
    sum(len(cases) for cases in controlled_plan.values()) * SAMPLES_PER_X
)

controlled_cmd = [
    sys.executable,
    str(controlled_script),
    "--profile", PROFILE,
    "--embeddings", str(SAMPLES_PER_X),
    "--timeout", str(TIMEOUT_S),
    "--seed", str(BASE_SEED),
]
controlled_rows = run_streamed_benchmark(
    controlled_cmd,
    total=controlled_total,
    description="Controlled Yamada families",
)


## 4. Heterogeneous connected-cubic benchmark

This is the complementary **generic-instance** test.

At every \(V\), the benchmark constructs 10 deterministic, pairwise **non-isomorphic**, connected, simple 3-regular graphs. Thus every sample satisfies

\[
E=\frac{3V}{2},
\]

but the actual graph topology differs from sample to sample. Each topology is embedded generically in 3-D, projected to a PD code outside the timed region, and that **same PD** is supplied to both frameworks.

The measured crossing count \(c\) is recorded rather than artificially fixed. Consequently this figure measures practical scaling across heterogeneous connected cubic instances at fixed \(V\), not a pure isolated-\(c\) law.


In [ ]:
random_cubic_script = ROOT / "dev" / "benchmark_topoly_random_cubic_ensemble.py"
random_cubic_module = _load_module(
    random_cubic_script,
    "kg_topoly_random_cubic_notebook_plan",
)
random_vertices = random_cubic_module.vertex_grid(PROFILE)
random_cubic_total = len(random_vertices) * SAMPLES_PER_X

random_cubic_cmd = [
    sys.executable,
    str(random_cubic_script),
    "--profile", PROFILE,
    "--samples-per-v", str(SAMPLES_PER_X),
    "--timeout", str(TIMEOUT_S),
    "--seed", str(BASE_SEED),
]
random_cubic_rows = run_streamed_benchmark(
    random_cubic_cmd,
    total=random_cubic_total,
    description="Non-isomorphic cubic graphs",
)


## 5. Acceptance checks and raw-data export

The notebook will not create paper figures unless the sampled data have the requested structure.

For the heterogeneous cubic ensemble, the benchmark itself performs exact pairwise NetworkX isomorphism checks before timing. The notebook additionally checks the recorded topology hashes, cubic degree constraint, connectivity flag, and sample counts.


In [ ]:
from collections import defaultdict

controlled_groups = defaultdict(list)
for row in controlled_rows:
    controlled_groups[(row["family"], row["size"])].append(row)

for key, group in controlled_groups.items():
    assert len(group) == SAMPLES_PER_X, (key, len(group), SAMPLES_PER_X)
    assert len({row["embedding"] for row in group}) == SAMPLES_PER_X
    assert len({row["embedding_hash"] for row in group}) == SAMPLES_PER_X
    assert len({row["embedding_seed"] for row in group}) == SAMPLES_PER_X

cubic_groups = defaultdict(list)
for row in random_cubic_rows:
    cubic_groups[int(row["V"])].append(row)

for V, group in cubic_groups.items():
    assert len(group) == SAMPLES_PER_X, (V, len(group), SAMPLES_PER_X)
    assert len({row["sample"] for row in group}) == SAMPLES_PER_X
    assert len({row["topology_instance_hash"] for row in group}) == SAMPLES_PER_X
    assert all(row["sample_kind"] == "topology" for row in group)
    assert all(row["nonisomorphic_ensemble_verified"] for row in group)
    assert all(row["connected"] for row in group)
    assert all(int(row["regular_degree"]) == 3 for row in group)
    assert all(int(row["E"]) == 3 * int(row["V"]) // 2 for row in group)

all_rows = controlled_rows + random_cubic_rows
for row in all_rows:
    if row["correctness"] == "PASS":
        assert row["knottedgraph_status"] == "ok"
        assert row["topoly_status"] == "ok"
        assert row["pd_hash"]

keys = list(dict.fromkeys(key for row in all_rows for key in row))
with raw_csv.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=keys)
    writer.writeheader()
    writer.writerows(all_rows)

print(f"PASS: controlled points contain {SAMPLES_PER_X} distinct embeddings/x.")
print(f"PASS: cubic points contain {SAMPLES_PER_X} pairwise non-isomorphic topologies/V.")
print("PASS: every paired successful KnottedGraph/Topoly evaluation passed Laurent-polynomial equivalence.")
print(f"wrote {len(all_rows)} sample-level records to {raw_csv}")


## 6. Paper-quality figures and confidence intervals

The plotting stage is intentionally separate from timing. You can rerun this cell after changing labels or formatting without repeating the expensive benchmark.

The random-cubic confidence interval is computed across **different graph topologies**. The controlled-family confidence intervals are computed across their independent geometric samples.


In [ ]:
plot_script = ROOT / "dev" / "plot_topoly_scaling.py"
plot_cmd = [
    sys.executable,
    str(plot_script),
    str(raw_csv),
    "--figure-dir", str(FIG),
    "--aggregate-csv", str(aggregate_csv),
]
print("Running:", " ".join(plot_cmd))
plot_process = subprocess.Popen(
    plot_cmd, cwd=ROOT, env=env, text=True, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, bufsize=1,
)
if plot_process.stdout is None:
    raise RuntimeError("Plot subprocess did not expose stdout.")
for line in plot_process.stdout:
    print(line, end="")
if plot_process.wait():
    raise RuntimeError("Paper-quality plot generation failed.")

expected_stems = [
    "topoly_vs_knottedgraph_crossings_fixed",
    "topoly_vs_knottedgraph_crossings_throughput",
    "topoly_vs_knottedgraph_edges",
    "topoly_vs_knottedgraph_vertices_k4",
    "topoly_vs_knottedgraph_prism_V",
    "topoly_vs_knottedgraph_prism_E",
    "topoly_vs_knottedgraph_random_cubic_V",
]
for stem in expected_stems:
    assert (FIG / f"{stem}.png").exists(), stem
    assert (FIG / f"{stem}.pdf").exists(), stem
assert aggregate_csv.exists()
print("PASS: all paper figure PNG/PDF pairs and aggregate confidence-interval CSV were created.")


## 7. Statistical interpretation

For each framework and x-axis point:

- **Controlled families:** one observation is one deterministic geometric embedding.
- **Random cubic family:** one observation is one pairwise non-isomorphic connected cubic graph topology.
- Any repeated stopwatch measurements inside a sample are first reduced to that sample's median.
- The plotted center is the median of the independent sample-level observations.
- The band is a nonparametric 95% bootstrap confidence interval for that median.
- Scaling fits require at least five successful independent samples at an x value.
- Timeout/error samples are reported as censored and are not replaced by the timeout threshold inside the confidence interval.

Therefore the random-cubic figure directly addresses instance-to-instance graph-topology variability, while the controlled figures retain their role as interpretable stress tests.
